# FP-Growth：比 Apriori 快 100 倍的频繁项集挖掘
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/关联规则 | 源文件：FP_Growth.py | 核心功能：FP-Tree 构建、无候选集挖掘、与 Apriori 的性能对比

## 概述

Apriori 需要多次扫描数据库并生成大量候选集，在大数据集上效率低下。FP-Growth（Frequent Pattern Growth）算法巧妙地用一棵**前缀树（FP-Tree）**压缩了整个交易数据库，只需扫描两次数据，不需要生成候选集，就能挖掘出所有频繁项集。

脚本在 500 条交易数据上对比了 FP-Growth 和 Apriori 的性能差异，结果通常显示 FP-Growth 快数倍到数十倍。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules, apriori
from mlxtend.preprocessing import TransactionEncoder
import time

## 数学原理

### 1. FP-Tree 数据结构

**代码对应**：FP-Growth 通过 FP-Tree 压缩事务数据库，避免多次扫描。

FP-Tree（Frequent Pattern Tree）是一种前缀树结构：
- 每个节点存储一个项及其计数
- 共享相同前缀的事务在树中共享路径
- 头指针表（Header Table）链接每个项的所有节点

**构建过程**：
1. 第一次扫描：统计每个项的支持度，删除非频繁项，按支持度降序排列
2. 第二次扫描：将每个事务按排序后的顺序插入树中，共享前缀路径

### 2. FP-Growth 算法

**核心思想**：将频繁项集挖掘问题分解为多个子问题。

对每个频繁项 $\alpha$：
1. 从 FP-Tree 中提取包含 $\alpha$ 的所有路径（条件模式基）
2. 用条件模式基构建 $\alpha$ 的**条件 FP-Tree**
3. 在条件 FP-Tree 上递归挖掘

### 3. 条件模式基与条件 FP-Tree

项 $\alpha$ 的**条件模式基**（Conditional Pattern Base）：以 $\alpha$ 结尾的所有前缀路径及其计数。

例如，如果项 $e$ 的条件模式基为 $\{(abc: 1), (ac: 2), (b: 1)\}$，则构建 $e$ 的条件 FP-Tree 时，$a$ 出现 3 次，$c$ 出现 3 次，$b$ 出现 2 次。如果 minsup = 2，则 $a$、$c$、$b$ 都进入条件 FP-Tree。

### 4. 与 Apriori 的对比

| 特性 | Apriori | FP-Growth |
|------|---------|-----------|
| 数据结构 | 候选项集 | FP-Tree |
| 数据库扫描次数 | $O(k)$（$k$ 为最大项集大小） | 2 次 |
| 候选生成 | 需要 | 不需要 |
| 时间复杂度 | $O(2^p)$ 最坏 | $O(n \cdot p)$ 通常 |
| 空间复杂度 | $O(p)$ | $O(n \cdot p)$（树存储） |

FP-Growth 通常比 Apriori 快一个数量级，但 FP-Tree 可能很大（极端情况下退化为链表）。

### 5. 支持度、置信度、提升度

FP-Growth 挖掘出的频繁项集同样使用支持度、置信度、提升度评估关联规则（与 Apriori 相同）。

额外的评估指标：

**杠杆率**（Leverage）：$\text{leverage}(X \Rightarrow Y) = P(X \cap Y) - P(X)P(Y)$

**确信度**（Conviction）：$\text{conviction}(X \Rightarrow Y) = \frac{1 - P(Y)}{1 - \text{conf}(X \Rightarrow Y)}$

Conviction 衡量规则预测错误的程度。$\text{conviction} = 1$ 表示 $X$ 和 $Y$ 独立，$\text{conviction} \to \infty$ 表示规则总是正确。

### 1. 构造较大规模交易数据

运行 1. 构造较大规模交易数据 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
items = ["牛奶", "面包", "黄油", "鸡蛋", "尿布", "啤酒", "可乐", "薯片", "巧克力", "苹果"]
n_transactions = 500

transactions = []
for _ in range(n_transactions):
    # 随机选择 2~5 个商品
    n_items = np.random.randint(2, 6)
    basket = list(np.random.choice(items, n_items, replace=False))
    # 添加一些关联模式
    if np.random.rand() < 0.4:
        basket.extend(["牛奶", "面包"])
    if np.random.rand() < 0.3:
        basket.extend(["尿布", "啤酒"])
    basket = list(set(basket))  # 去重
# --- 继续 ---
    transactions.append(basket)

print(f"交易总数: {len(transactions)}")
print(f"平均每笔交易商品数: {np.mean([len(t) for t in transactions]):.1f}")

### 2. 数据编码

运行 2. 数据编码 的代码，观察算法在该环节的行为。

In [ ]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)
print(f"交易矩阵形状: {df.shape}")

### 3. FP-Growth 挖掘

运行 3. FP-Growth 挖掘 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== FP-Growth (min_support=0.15) ===")
t0 = time.time()
fi_fp = fpgrowth(df, min_support=0.15, use_colnames=True)
t_fp = time.time() - t0
print(f"耗时: {t_fp:.4f}s")
# --- 输出结果 ---
print(f"频繁项集数: {len(fi_fp)}")
print(fi_fp.sort_values("support", ascending=False).head(10).to_string())

### 4. 与 Apriori 对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== FP-Growth vs Apriori 性能对比 ===")
for ms in [0.1, 0.15, 0.2]:
    t0 = time.time()
    fi_ap = apriori(df, min_support=ms, use_colnames=True)
    t_ap = time.time() - t0

    t0 = time.time()
    fi_fp2 = fpgrowth(df, min_support=ms, use_colnames=True)
    t_fp2 = time.time() - t0

    print(f"  min_support={ms}: Apriori={t_ap:.4f}s ({len(fi_ap)} 项集), "
          f"FP-Growth={t_fp2:.4f}s ({len(fi_fp2)} 项集)")

### 5. 生成关联规则

运行 5. 生成关联规则 的代码，观察算法在该环节的行为。

In [ ]:
rules = association_rules(fi_fp, metric="confidence", min_threshold=0.5)
print(f"\n=== 关联规则 (confidence≥0.5) ===")
print(f"共 {len(rules)} 条规则")

# 按 lift 排序
rules_lift = rules.sort_values("lift", ascending=False)
for _, row in rules_lift.head(8).iterrows():
    ant = ", ".join(row["antecedents"])
    con = ", ".join(row["consequents"])
    print(f"  {ant} → {con}")
# --- 输出结果 ---
    print(f"    S={row['support']:.3f}, C={row['confidence']:.3f}, L={row['lift']:.3f}")

### 6. 提升度 vs 置信度过滤

运行 6. 提升度 vs 置信度过滤 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 高提升度高置信度规则 ===")
high_quality = rules[(rules["lift"] > 1.5) & (rules["confidence"] > 0.6)]
print(f"满足条件 (lift>1.5 & confidence>0.6) 的规则: {len(high_quality)} 条")
for _, row in high_quality.iterrows():
    ant = ", ".join(row["antecedents"])
# --- 继续 ---
    con = ", ".join(row["consequents"])
    print(f"  {ant} → {con} (L={row['lift']:.3f}, C={row['confidence']:.3f})")

### 7. 算法原理

运行 7. 算法原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== FP-Growth 算法原理 ===")
print("1. 第一次扫描：统计每个项的频率，按频率降序排序")
print("2. 第二次扫描：构建 FP-Tree（前缀树）")
print("   - 每条交易按频率排序后插入树中")
print("   - 共享公共前缀，节省空间")
# --- 输出结果 ---
print("3. 递归挖掘：从每个频繁项出发，提取条件模式基")
print("   - 构建条件 FP-Tree")
print("   - 在条件树上递归挖掘")

print("\n=== FP-Growth vs Apriori ===")
print("Apriori: 需要多次扫描数据 + 生成候选集 → 慢")
print("FP-Growth: 只扫描 2 次 + 不需要候选集 → 快")
print("FP-Growth 在大数据集上可快 10~100 倍")

print("\n=== FP-Growth 要点 ===")
print("- 优点：不需要生成候选集，只需扫描数据库两次，速度快")
print("- 缺点：FP-Tree 可能很大，内存消耗高")
print("- min_support 是最关键的参数")
print("- 与 Apriori 产生完全相同的结果")

## 关键代码解释

### 性能对比

```python
for ms in [0.1, 0.15, 0.2]:
    fi_ap = apriori(df, min_support=ms, use_colnames=True)
    fi_fp2 = fpgrowth(df, min_support=ms, use_colnames=True)
```

FP-Growth 和 Apriori 产生**完全相同的结果**，只是计算效率不同。min_support 越低，差距越大。

### 高质量规则过滤

```python
high_quality = rules[(rules["lift"] > 1.5) & (rules["confidence"] > 0.6)]
```

实际应用中通常用多个条件组合过滤，保留真正有价值的规则。

## 使用示例

```python
from mlxtend.frequent_patterns import fpgrowth, association_rules
fi = fpgrowth(df, min_support=0.15, use_colnames=True)
rules = association_rules(fi, metric="lift", min_threshold=1.0)
```

## 注意事项

1. **与 Apriori 结果完全相同**，只是速度更快
2. **FP-Tree 可能占用大量内存**：数据维度很高时注意内存
3. **min_support 仍是关键参数**
4. **适合大规模数据**：小数据集两者差别不大

## 总结与延伸

以上代码展示了 **FP Growth** 的完整流程。

**解读要点**：
- 关注各项评估指标的变化趋势
- 对比不同参数配置下的性能差异
- 注意模型的训练时间和资源消耗

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **FP-Growth 的原理**：第一次扫描统计频率并排序，第二次扫描构建 FP-Tree，然后递归挖掘条件模式基
- **Eclat 算法**：基于等价类的垂直数据格式挖掘，也是 Apriori 的高效替代
- **在线关联规则挖掘**：增量更新 FP-Tree 而不需重建
